# Run the benchmark — mouse MERFISH MOp cortex

Two steps:

1. **Step 1 — preprocess once.** Produces input format all the methods
   need, so all six methods see the same cells, genes, filtering and held-out
   cell type. Run once per setting.
2. **Step 2 — run one method.** Each method is independent; run only the ones
   you want.

| role | file |
|---|---|
| **Reference** | `merfish_split1.h5ad` |
| **Target** | `merfish_split2.h5ad` |

## Methods

| method | code | input format from stage 1 |
|---|---|---|
| **NovAST** | **installed package** (`pip install -e ..`) | `*_normalized.h5ad` |
| STELLAR | vendored `scripts/STELLAR_py` | `loader_output.pkl` (+ spatial graph) |
| scBOL | vendored `scripts/scBOL` | `loader_output.pkl` (+ spatial graph) |
| SpatialID | vendored `scripts/SpatialID` | `*_preprocessed.h5ad` (counts) |
| Tangram | installed `tangram-sc` + wrapper | `*_preprocessed.h5ad` (counts) |
| SingleR | R / Bioconductor | `SingleR_input/` (.mtx + .csv) |


## Configuration

### The `SETTING` — which cell type is held out from the Reference

A *setting* picks one cell type to drop from the **Reference** so it
survives **only in the Target**. That makes it an artificial **novel**
cell type: the methods can't match it to any known label, they have to discover
it. This is the core stress-test of the benchmark.

| `SETTING` | display name | dropped from the Reference | selected by |
|---|---|---|---|
| `none` | — | nothing — all types shared | — |
| `most` | **Most Abundant** | the most frequent Target cell type | `freq.idxmax()` |
| `least` | **Least Abundant** | the rarest Target cell type | `freq.idxmin()` |
| `closest` | **Most Mixed** | the type sitting *among* the others — **smallest** mean distance between UMAP centroids | `nanargmin(mean_dist)` |
| `furthest` | **Most Separated** | the type standing *apart* — **largest** mean distance between UMAP centroids | `nanargmax(mean_dist)` |

### This notebook uses `most` — the **Most Abundant** case — as its example

The most frequent cell type in the Target is withheld from the Reference, so a large,
well-populated group becomes the novel type. Change `SETTING` below to run any other case. Each setting writes to its own
`results/merfish_split_<SETTING>/`, so they never collide — you can run all five
and compare.

In [ ]:
import os, sys, subprocess, json, time

# ---- paths: this benchmark package -----------------------------------------
# BASE is this benchmark directory. Jupyter starts the kernel in the notebook's
# folder, so the working directory is benchmark/ when the notebook is run from
# here. If you run it from elsewhere, set BASE to the benchmark directory.
BASE    = os.getcwd()
assert os.path.isdir(os.path.join(BASE, "scripts")), (
    f"Expected to run from the benchmark/ directory, but cwd is {BASE}. "
    "Set BASE to the benchmark directory explicitly."
)
SCRIPTS = os.path.join(BASE, "scripts")   # pipeline code: main.py + one folder per method
MAIN    = os.path.join(SCRIPTS, "main.py")  # entry point for BOTH stages (--stage preprocess|run)
SINGLER = os.path.join(SCRIPTS, "SingleR")  # SingleR is an R workflow outside main.py -> called directly
RESULTS = os.path.join(BASE, "results")     # every run writes under results/<NAME>/
PYTHON  = sys.executable                    # this kernel's interpreter; handed to every subprocess
                                            # so children run in the same environment
os.makedirs(RESULTS, exist_ok=True)

# ---- input data: the MERFISH MOP split -------------------------------------
# split1 = Reference (labelled; what the methods learn from)
# split2 = Target    (to annotate; holds the held-out novel type)
# Set DATA_DIR to the local folder holding the two .h5ad files.
DATA_DIR      = "/path/to/merfish_mop"
RAW_REFERENCE = os.path.join(DATA_DIR, "merfish_split1.h5ad")
RAW_TARGET    = os.path.join(DATA_DIR, "merfish_split2.h5ad")

# ---- experiment ------------------------------------------------------------
SETTING = "most"   # which cell type to hold out of the Reference (see the table above).
                   # "most" = Most Abundant -> the worked example in this notebook.
                   # options: none | most | least | closest | furthest
NAME    = f"merfish_split_{SETTING}"    # run id -- keeps each setting's results separate
SAVEDIR = RESULTS + "/"                 # main.py joins savedir + name internally
RUN_DIR = os.path.join(RESULTS, NAME)   # where stage 1 + stage 2 outputs actually land

print("setting:", SETTING, "-> holding out the Most Abundant Target cell type")
print("run dir:", RUN_DIR)
assert os.path.exists(MAIN) and os.path.exists(RAW_REFERENCE) and os.path.exists(RAW_TARGET)


In [ ]:
def sh(cmd):
    """Run a command, streaming its output."""
    print(">>>", " ".join(str(c) for c in cmd), "\n", flush=True)
    t0 = time.time()
    subprocess.run([str(c) for c in cmd], check=True)
    print(f"\n[done in {time.time()-t0:.1f}s]", flush=True)

def run_main(extra):
    sh([PYTHON, "-u", MAIN] + extra)


---
# Step 1 — preprocess (once per setting)

Does the work **once** so every method is comparable:

- intersect genes between reference and target
- `normalize_total` → `log1p` → `scale`
- apply the controlled filter (drop Target cells whose label is absent from the Reference)
- **then** optionally drop the held-out cell type from Reference
- build the spatial kNN graph

### Outputs (all formats)

| file | consumed by |
|---|---|
| `train/target_normalized.h5ad` | **NovAST** (X = normalized) |
| `loader_output.pkl` | STELLAR, scBOL (X = normalized + graph) |
| `train/target_preprocessed.h5ad` | SpatialID, Tangram (X = counts) |
| `inverse_dict_reference/test.pkl` | label↔name maps (all methods) |
| `preprocess_args.json` | replayed by stage 2 |
| `SingleR_input/` | SingleR (added by 1b below) |


In [ ]:
# --- 1a: shared preprocess -> h5ad (both flavours), loader_output.pkl, graph ---
pre_args = [
    "--stage", "preprocess",
    "--savedir", SAVEDIR,
    "--name", NAME,
    "--reference_path", RAW_REFERENCE,
    "--target_path",  RAW_TARGET,
    "--region_name_reference", "slice",
    "--region_name_target",  "slice",
    "--celltype_name_reference", "cell_type",
    "--celltype_name_target",  "cell_type",
    "--region_name_reference_select", "brain_section_label",
    "--region_name_target_select",  "brain_section_label",
    # spatial graph (STELLAR / scBOL)
    "--num-heads", 22,
    "--num_parts_reference", 10, "--num_parts_target", 10,
    "--distance_thres_reference", 50, "--distance_thres_target", 50,
]
if SETTING != "none":
    pre_args += ["--remove-celltype", "--remove-celltype-type", SETTING]
    # reproducibility: pin the held-out type for closest/furthest, e.g.
    # pre_args += ["--remove_celltype_name", "<cell type>"]

run_main(pre_args)


In [ ]:
# --- 1b: SingleR's format (.mtx + .csv) derived from the counts h5ads ---
sh([PYTHON, os.path.join(SINGLER, "singler_io.py"), "prepare", "--savedir", RUN_DIR])


In [ ]:
# What stage 1 produced
for f in sorted(os.listdir(RUN_DIR)):
    print(" ", f)
with open(os.path.join(RUN_DIR, "preprocess_stats.json")) as fh:
    print("\nstats:", json.load(fh))


---
# Step 2 — run one method

Each cell is independent, so run only what you need. Every method:

- runs `--rounds` seeds (**default 10**; `rounds=1` below for a quick smoke test),
- is **idempotent per seed** — finished seeds are skipped, so re-running is safe,
- writes to `results/<NAME>/<Method>/`.

Step 2 replays `preprocess_args.json`, so it inherits step 1's settings.


In [ ]:
def run_method(method, rounds=None):
    extra = ["--stage", "run", "--savedir", SAVEDIR, "--name", NAME, "--method", method]
    if rounds is not None:
        extra += ["--rounds", rounds]
    run_main(extra)


### NovAST — using the installed package

NovAST is not included directly in this repository. Instead, `scripts/NovAST_py/run_benchmark.py` serves as a lightweight wrapper around the installed NovAST package. It loads the stage 1 `*_normalized.h5ad` file and passes it to the package’s `run_NovAST(...)` function.

Installation instructions are provided in `scripts/NovAST_py/README.md`.

By default, the benchmark uses the hyperparameters defined in NovAST’s packaged `default_config.yaml`.

In [ ]:
# Confirm which NovAST is installed before running it
import NovAST
print("NovAST from:", NovAST.__file__)
print("exports    :", [x for x in dir(NovAST) if not x.startswith("_")])


In [ ]:
run_method("NovAST", rounds=1)


In [ ]:
# Optional: override values from the package's default_config.yaml.
# Any parameter not listed in `overrides` will continue to use the packaged default.

# Uncomment the following line when calling NovAST directly from Python:
# from NovAST_py.run_benchmark import NovAST_main

# Example ablation run with custom hyperparameters:
# NovAST_main(
#     args,
#     overrides={
#         "gamma": 2,
#         "beta": 0.3,
#         "mmf_k": 100,
#         "mmf_margin": 2,
#     },
# )


### The other GPU methods

In [ ]:
run_method("STELLAR",   rounds=1)
run_method("scBOL",     rounds=1)
run_method("SpatialID", rounds=1)
run_method("Tangram",   rounds=1)

### SingleR — the R baseline

SingleR is not driven by `main.py`; it's a file-based R workflow (no reticulate).
`prepare` already ran in **stage 1b**, so only two steps remain. **CPU-only** — no
GPU needed.

Outputs land in `results/<NAME>/SingleR/{prediction.npy, ground_truth.npy}`.

In [ ]:
# SingleR is DETERMINISTIC -- no seeds, no rounds. run_singler.R classifies once
# ("verified identical across seeds") and singler_io.py takes only --savedir, so
# notebook 02 scores SingleR once rather than averaging over seeds.

R_MODULE = "r/4.4.0-c4wv"    # same module the original benchmark sbatch used

def sh_with_R(cmd):
    """Run a command with the Lmod R module loaded.

    The kernel's PATH has no R, so we hand the command to a bash shell that
    first initialises Lmod and loads R. This mirrors the original sbatch:
        source <lmod init> && module load r/4.4.0-c4wv && Rscript ...
    """
    inner = " ".join(f"'{c}'" for c in map(str, cmd))
    script = (
        "source /usr/share/lmod/lmod/init/bash 2>/dev/null || "
        "source /etc/profile.d/z00_lmod.sh 2>/dev/null || "
        "source /etc/profile 2>/dev/null; "
        f"module load {R_MODULE} && exec {inner}"
    )
    sh(["bash", "-c", script])

# 2. R: run SingleR  ->  results/<NAME>/SingleR/singler_predictions.csv
sh_with_R(["Rscript", os.path.join(SINGLER, "run_singler.R"), "--savedir", RUN_DIR])

# 3. Python: csv -> prediction.npy + ground_truth.npy  (plain kernel python)
sh([PYTHON, os.path.join(SINGLER, "singler_io.py"), "convert", "--savedir", RUN_DIR])


---
## What ran?

In [ ]:
for m in ["NovAST", "STELLAR", "scBOL", "SpatialID", "Tangram", "SingleR"]:
    d = os.path.join(RUN_DIR, m)
    if not os.path.isdir(d):
        print(f"{m:10s} -- not run")
        continue
    seeds = sorted(x for x in os.listdir(d) if x.startswith("seed"))
    print(f"{m:10s} {len(seeds):2d} seed dir(s)" if seeds else f"{m:10s} present (no seed dirs)")


After the benchmarking runs are complete, open **`02_evaluate_merfish_split.ipynb`** to evaluate all successfully completed methods.

### Running the benchmark on SLURM

Running all seeds and methods sequentially can be time-consuming. For full-scale experiments, submit the preprocessing step first and then launch a separate SLURM job for each benchmarking method after preprocessing completes.

For example:

```bash
sbatch --job-name=pre preprocess.sh
sbatch --dependency=afterok:<preprocessing_job_id> run_NovAST.sh
```

Replace `<preprocessing_job_id>` with the job ID returned by the first `sbatch` command. The `afterok` dependency ensures that the NovAST job starts only after preprocessing finishes successfully.

Each method-specific job script only needs to run the corresponding command printed by the notebook:

```bash
python main.py --stage run --method <METHOD>
```

Replace `<METHOD>` with the method name, and submit one job for each method you want to benchmark.